# CDK20_HUMAN: 相同蛋白の探索と既知活性化合物の収集

CDK20_HUMANはまだX線結晶構造が解かれていない。そこで、

1. CDK20自体の既知の活性化合物をChEMBLから収集する
2. UniProt/BLASTでCDK20に配列が近い蛋白を探す
3. 見つかった蛋白ごとにPDBエントリ数・ChEMBL活性化合物数を集計し、この研究で参照蛋白として活用できそうなものをランキングする


In [1]:
from pathlib import Path

TARGET = "CDK20_HUMAN"
OUTDIR = Path("data/cdk20_investigation")
OUTDIR.mkdir(parents=True, exist_ok=True)

# ChEMBL Web APIが障害等で使えない場合のフォールバック(ChEMBL公式配布のSQLite)
CHEMBL_DB = Path("data/chembl/chembl_36.db")

In [2]:
from idmap import resolve_uniprot_accession

accession = resolve_uniprot_accession(TARGET)
print(accession)

02:51:42 [idmap.identifiers] Resolving UniProt entry name CDK20_HUMAN -> accession ...
02:52:04 [idmap.identifiers]   -> Q8IZL9


Q8IZL9


## 0. AlphaFoldの予測構造をダウンロードする

CDK20_HUMANはまだX線結晶構造が解かれていないため、AlphaFold DBが提供する予測構造(cif)を先にダウンロードしておく。


In [3]:
from afdb import fetch_structure

AFDB_DIR = OUTDIR / "alphafold"
afdb_output = AFDB_DIR / f"{accession}.cif"

if afdb_output.exists():
    print(f"{afdb_output}: already exists, skipping")
else:
    fetch_structure(accession, afdb_output)

data/cdk20_investigation/alphafold/Q8IZL9.cif: already exists, skipping


## 1. CDK20自体の既知活性化合物を収集(ChEMBL)

構造とは独立に、CDK20_HUMANに対する既知の活性化合物(pChEMBL値あり)をChEMBLから取得する。


In [4]:
import pandas as pd
from chembl import local as chembl_local

chembl_target_id = chembl_local.resolve_target_chembl_id(accession, CHEMBL_DB)
activities = chembl_local.fetch_activities(chembl_target_id, CHEMBL_DB)

compounds_df = pd.DataFrame(activities)
if not compounds_df.empty:
    compounds_df = compounds_df[
        ["molecule_chembl_id", "molecule_pref_name", "canonical_smiles",
         "standard_type", "standard_value", "standard_units", "pchembl_value",
         "assay_chembl_id", "document_chembl_id"]
    ].sort_values("pchembl_value", ascending=False).reset_index(drop=True)
compounds_df.to_csv(OUTDIR / "known_compounds.csv", index=False)
compounds_df

02:52:07 [chembl.local] Resolving UniProt accession Q8IZL9 -> ChEMBL target id (local DB) ...
02:52:07 [chembl.local]   -> CHEMBL3559690
02:52:07 [chembl.local] Fetching activities for target CHEMBL3559690 (pChEMBL value required, local DB) ...
02:52:07 [chembl.local]   -> 1 activities fetched for CHEMBL3559690


,molecule_chembl_id,molecule_pref_name,canonical_smiles,standard_type,standard_value,standard_units,pchembl_value,assay_chembl_id,document_chembl_id
0,CHEMBL4445812,None,O=C(Nc1ccc2cc(C(=O)Nc3nccs3)ccc2c1)c1ccc(Cl)c(...,Kd,8020,nM,5.1,CHEMBL4375310,CHEMBL4373706


In [5]:
from uniprot.entry import fetch_fasta

fasta = fetch_fasta(accession).decode()
sequence = "".join(line for line in fasta.splitlines() if not line.startswith(">"))
print(fasta)
print(f"sequence length: {len(sequence)}")

02:52:07 [uniprot.entry] Fetching UniProt FASTA for Q8IZL9 ...


>sp|Q8IZL9|CDK20_HUMAN Cyclin-dependent kinase 20 OS=Homo sapiens OX=9606 GN=CDK20 PE=1 SV=1
MDQYCILGRIGEGAHGIVFKAKHVETGEIVALKKVALRRLEDGFPNQALREIKALQEMED
NQYVVQLKAVFPHGGGFVLAFEFMLSDLAEVVRHAQRPLAQAQVKSYLQMLLKGVAFCHA
NNIVHRDLKPANLLISASGQLKIADFGLARVFSPDGSRLYTHQVATRWYRAPELLYGARQ
YDQGVDLWSVGCIMGELLNGSPLFPGKNDIEQLCYVLRILGTPNPQVWPELTELPDYNKI
SFKEQVPMPLEEVLPDVSPQALDLLGQFLLYPPHQRIAASKALLHQYFFTAPLPAHPSEL
PIPQRLGGPAPKAHPGPPHIHDFHVDRPLEESLLNPELIRPFILEG

sequence length: 346


## 2. UniProt/BLASTでCDK20に近い蛋白を探す

swissprot(UniProtKB/Swiss-Prot、審査済みエントリ)データベースに対し、Homo sapiensに限定してBLASTサーチを行い、
CDK20_HUMANに配列が近い蛋白(UniProt accession)を集める。配列同一性(identity)が40%を割る蛋白は
遠縁とみなし対象から除外する。


In [6]:
from blastsearch import run_cached_blast

hits = run_cached_blast(
    sequence,
    cache_dir=OUTDIR,
    program="blastp",
    database="swissprot",
    entrez_query="Homo sapiens[Organism]",
)
print(f"{len(hits)} hit(s)")


02:52:14 [blastsearch.cache] Using cached BLAST hits: data/cdk20_investigation/blast_hits.pkl


101 hit(s)


In [7]:
import pandas as pd
from blastsearch import best_hit_per_accession, format_evalue
from uniprot.entry import fetch_entry_names

# UniProt accessionごとに最良ヒット(evalue最小)だけを残す。CDK20_HUMAN自身は除く。
unique_hits = best_hit_per_accession(hits, exclude_accession=accession)

hits_df = pd.DataFrame(unique_hits)

IDENTITY_MIN = 40.0
hits_df = hits_df[hits_df["identity"] >= IDENTITY_MIN].reset_index(drop=True)

entry_names = fetch_entry_names(hits_df["accession"].tolist())
hits_df["entry_name"] = hits_df["accession"].map(entry_names)
hits_df["coverage"] = (hits_df["align_length"] / len(sequence) * 100).round(1)
hits_df = hits_df[["accession", "entry_name", "identity", "coverage", "align_length", "evalue", "bit_score"]]
hits_df.to_csv(OUTDIR / "blast_hits.csv", index=False)
print(f"unique candidate proteins (identity >= {IDENTITY_MIN}%): {len(hits_df)}")

display_df = hits_df.head(20).copy()
display_df["identity"] = display_df["identity"].round(2)
display_df["bit_score"] = display_df["bit_score"].round().astype(int)
display_df["evalue"] = display_df["evalue"].apply(format_evalue)
display_df


02:52:14 [uniprot.entry] Fetching UniProt entry names for 9 accessions ...
02:52:19 [uniprot.entry]   -> 9 entry names resolved


unique candidate proteins (identity >= 40.0%): 9


,accession,entry_name,identity,coverage,align_length,evalue,bit_score
0,Q00526,CDK3_HUMAN,45.28,88.7,307,6.8e-86,261
1,P06493,CDK1_HUMAN,43.10,83.8,290,5.0e-80,246
2,P50613,CDK7_HUMAN,43.14,88.4,306,6.8e-80,247
3,P24941,CDK2_HUMAN,43.77,85.8,297,1.1e-79,245
4,Q00535,CDK5_HUMAN,46.02,83.5,289,5.6e-79,243
5,P21127,CD11B_HUMAN,42.16,88.4,306,3.2e-73,241
6,Q9UQ88,CD11A_HUMAN,41.83,88.4,306,7.3e-73,240
7,P50750,CDK9_HUMAN,40.00,89.6,310,1.2e-68,219
8,O76039,CDKL5_HUMAN,40.00,85.3,295,2.8e-60,207


## 3. 見つかった蛋白ごとにPDBエントリ数・ChEMBL化合物数を集計してランキングする

上位ヒットについて、UniProtのPDB相互参照からPDBエントリ数を、ChEMBLから既知活性化合物数(ユニークな化合物数)を
集計する。両方が揃っている蛋白ほど、この研究(ドッキングテンプレート+SAR参照)で活用しやすいと考えられる。


In [8]:
TOP_N = 40
candidates_df = hits_df.head(TOP_N).copy()

In [9]:
import pickle

import requests
from uniprot.entry import fetch_protein_info
from chembl import local as chembl_local

RANKING_CACHE = OUTDIR / "protein_ranking.csv"
PDB_STRUCTURES_CACHE = OUTDIR / "pdb_structures.pkl"

if RANKING_CACHE.exists() and PDB_STRUCTURES_CACHE.exists():
    print(f"Using cached ranking: {RANKING_CACHE}")
    ranking_df = pd.read_csv(RANKING_CACHE)
    with open(PDB_STRUCTURES_CACHE, "rb") as f:
        pdb_structures_by_accession = pickle.load(f)
else:
    extra_rows = []
    pdb_structures_by_accession = {}
    for _, row in candidates_df.iterrows():
        acc = row["accession"]
        try:
            info = fetch_protein_info(acc)
            pdb_count = len(info["pdb_structures"])
            pdb_structures_by_accession[acc] = info["pdb_structures"]
        except requests.exceptions.RequestException as e:
            print(f"  [skip] {acc}: failed to fetch UniProt info ({e})")
            continue

        chembl_target_id = None
        try:
            chembl_target_id = chembl_local.resolve_target_chembl_id(acc, CHEMBL_DB)
            compounds = chembl_local.fetch_activities(chembl_target_id, CHEMBL_DB)
            compound_count = len({a["molecule_chembl_id"] for a in compounds})
        except ValueError:
            compound_count = 0  # ChEMBL target(SINGLE PROTEIN)が見つからない

        extra_rows.append({
            "accession": acc,
            "pdb_count": pdb_count,
            "chembl_target_id": chembl_target_id,
            "compound_count": compound_count,
        })

    extra_df = pd.DataFrame(extra_rows)
    ranking_df = candidates_df.merge(extra_df, on="accession").sort_values(
        ["pdb_count", "compound_count"], ascending=False
    ).reset_index(drop=True)
    ranking_df.to_csv(RANKING_CACHE, index=False)
    with open(PDB_STRUCTURES_CACHE, "wb") as f:
        pickle.dump(pdb_structures_by_accession, f)
    print(f"Saved ranking to {RANKING_CACHE}")

Using cached ranking: data/cdk20_investigation/protein_ranking.csv


In [10]:
from blastsearch import format_evalue

ranking_display_df = ranking_df[
    ["accession", "entry_name", "chembl_target_id", "identity", "coverage",
     "align_length", "evalue", "bit_score", "pdb_count", "compound_count"]
].copy()
ranking_display_df["identity"] = ranking_display_df["identity"].round(2)
ranking_display_df["bit_score"] = ranking_display_df["bit_score"].round().astype(int)
ranking_display_df["evalue"] = ranking_display_df["evalue"].apply(format_evalue)
ranking_display_df


,accession,entry_name,chembl_target_id,identity,coverage,align_length,evalue,bit_score,pdb_count,compound_count
0,P24941,CDK2_HUMAN,CHEMBL301,43.77,85.8,297,1.1e-79,245,512,2618
1,P50613,CDK7_HUMAN,CHEMBL3055,43.14,88.4,306,6.8e-80,247,53,627
2,P50750,CDK9_HUMAN,CHEMBL3116,40.00,89.6,310,1.2e-68,219,28,1741
3,P06493,CDK1_HUMAN,CHEMBL308,43.10,83.8,290,5.0e-80,246,13,1475
4,Q00535,CDK5_HUMAN,CHEMBL4036,46.02,83.5,289,5.6e-79,243,10,717
5,Q00526,CDK3_HUMAN,CHEMBL4442,45.28,88.7,307,6.8e-86,261,2,47
6,P21127,CD11B_HUMAN,CHEMBL5808,42.16,88.4,306,3.2e-73,241,2,20
7,Q9UQ88,CD11A_HUMAN,CHEMBL5416,41.83,88.4,306,7.3e-73,240,0,48


## 4. 解像度1.5Å未満のX線構造を一括ダウンロードする

セクション2で集計したPDBエントリのうち、実験手法がX線結晶構造解析(`method == "X-ray"`)かつ
解像度1.5Å未満のものをすべてダウンロードする。蛋白(entry_name)ごとにサブディレクトリへ保存し、
既にファイルが存在する場合はスキップするので、繰り返し実行しても差分だけ取得される。


In [11]:
from rcsb import fetch_structure
from uniprot.entry import parse_resolution

RESOLUTION_CUTOFF = 1.5
STRUCT_DIR = OUTDIR / "structures"

entry_name_by_accession = candidates_df.set_index("accession")["entry_name"].to_dict()

# (accession, entry_name, pdb_id, resolution) のリストに集約(解像度の良い順)
targets = []
for acc, structures in pdb_structures_by_accession.items():
    for s in structures:
        if s["method"] != "X-ray":
            continue
        res = parse_resolution(s["resolution"])
        if res is None or res >= RESOLUTION_CUTOFF:
            continue
        targets.append((acc, entry_name_by_accession.get(acc, acc), s["id"], res))
targets.sort(key=lambda t: t[3])

total = len(targets)
print(f"X-ray structures with resolution < {RESOLUTION_CUTOFF} A: {total}")

downloaded = 0
skipped = 0
for i, (acc, entry_name, pdb_id, res) in enumerate(targets, start=1):
    subdir = STRUCT_DIR / entry_name
    subdir.mkdir(parents=True, exist_ok=True)
    output = subdir / f"{pdb_id}.cif"
    if output.exists():
        print(f"[{i}/{total}] {entry_name}/{pdb_id}: already exists, skipping")
        skipped += 1
        continue
    print(f"[{i}/{total}] {entry_name}/{pdb_id}: downloading (resolution={res:.2f} A) ...")
    fetch_structure(pdb_id, output)
    downloaded += 1

print(f"Done: {downloaded} downloaded, {skipped} already present, {total} total, saved under {STRUCT_DIR}")


X-ray structures with resolution < 1.5 A: 30
[1/30] CDK2_HUMAN/6Q4G: already exists, skipping
[2/30] CDK2_HUMAN/6Q49: already exists, skipping
[3/30] CDK2_HUMAN/6Q4H: already exists, skipping
[4/30] CDK2_HUMAN/6Q48: already exists, skipping
[5/30] CDK2_HUMAN/6Q4J: already exists, skipping
[6/30] CDK2_HUMAN/6Q4E: already exists, skipping
[7/30] CDK2_HUMAN/6Q4K: already exists, skipping
[8/30] CDK2_HUMAN/6Q4D: already exists, skipping
[9/30] CDK2_HUMAN/6Q3B: already exists, skipping
[10/30] CDK2_HUMAN/6Q4I: already exists, skipping
[11/30] CDK2_HUMAN/6Q4B: already exists, skipping
[12/30] CDK2_HUMAN/6Q4A: already exists, skipping
[13/30] CDK2_HUMAN/9GNO: already exists, skipping
[14/30] CDK2_HUMAN/6Q3F: already exists, skipping
[15/30] CDK2_HUMAN/6Q4F: already exists, skipping
[16/30] CDK2_HUMAN/4EK4: already exists, skipping
[17/30] CDK2_HUMAN/4FKL: already exists, skipping
[18/30] CDK2_HUMAN/2R3I: already exists, skipping
[19/30] CDK2_HUMAN/6Q3C: already exists, skipping
[20/30] CDK2_H

## 5. ATP結合部位周辺残基でAlphaFold予測構造にアラインする

fpocketでCDK20のAlphaFold予測構造(セクション0でダウンロード)のポケットを検出する。fpocketのスコア
(druggability等)は「ATP結合部位かどうか」を直接表さないため、次の2種類のアンカー残基を根拠にATP結合部位を
特定する: (1) キナーゼドメインの保存モチーフ(P-loop/触媒Lys/HRD/DFG)を配列から検出したもの、
(2) ダウンロード済みの相同蛋白のX線構造のうち共結晶化リガンド(ATP拮抗阻害剤)を含むものについて、
その接触残基をCDK20の番号にマッピングし、一定割合以上の構造で再現されたものだけを採用したもの。
この2つを統合したアンカー残基を最もよく含むポケットをATP結合部位とする。
ダウンロードした相同蛋白のX線構造(水分子を除く)を、配列アラインメントで対応付けたそのATP結合部位周辺残基の
CA原子だけを使ってこの基準構造に重ね合わせる(チェーンIDは変更しない)。整列後の構造は各蛋白ディレクトリ下の
`aligned`サブディレクトリに`<PDB ID>_aligned.cif`として書き出す。


In [12]:
from kinasemotifs import find_kinase_motifs
from pocket import run_fpocket
from seqextract import get_chain_sequences
from structio import parse_structure

POCKET_DIR = OUTDIR / "alphafold_pocket"
pockets = run_fpocket(afdb_output, POCKET_DIR)
print(f"detected {len(pockets)} pocket(s) in the AlphaFold model")

target_atoms = parse_structure(afdb_output)
target_chains = get_chain_sequences(target_atoms)
if len(target_chains) != 1:
    raise ValueError(f"expected a single chain in the AlphaFold model, got {[c.chain_id for c in target_chains]}")
target_chain = target_chains[0]
target_chain_id = target_chain.chain_id

# --- アンカー1: キナーゼドメインの保存モチーフ(P-loop/触媒Lys/HRD/DFG/DFG+1) ---
# fpocketのスコア(druggability等)は「ATP結合部位かどうか」を直接表さない。特にリガンドを
# 含まないAlphaFold予測構造では、他の表面ポケットがスコアで上回ることが普通に起こる。そこで
# キナーゼドメインの保存モチーフ(ATP結合部位を構成することが構造生物学的に確立している残基)を
# アンカーとする(モチーフの生物学的根拠はkinasemotifs.find_kinase_motifsのdocstring参照)。
motifs = find_kinase_motifs(target_chain.sequence)
motif_anchor_resnums = motifs.anchor_resnums
print(f"motif-based anchor residues: {sorted(motif_anchor_resnums)}")


02:52:20 [pocket.fpocket] Running fpocket on data/cdk20_investigation/alphafold/Q8IZL9.cif
02:52:21 [pocket.fpocket] Done: found 28 pocket(s) in data/cdk20_investigation/alphafold/Q8IZL9.cif
02:52:21 [kinasemotifs.motifs] P-loop (GxGxxG) motif: resnum 11-16 (GEGAHG)
02:52:21 [kinasemotifs.motifs] Catalytic Lys (VAxK) motif: resnum 33 (VALK)
02:52:21 [kinasemotifs.motifs] Catalytic loop (HRD) motif: resnum 125-127
02:52:21 [kinasemotifs.motifs] DFG motif: resnum 145-147
02:52:21 [kinasemotifs.motifs] DFG+1 (back pocket wall): resnum 148


detected 28 pocket(s) in the AlphaFold model
motif-based anchor residues: [11, 12, 13, 14, 15, 16, 33, 125, 126, 127, 145, 146, 147, 148]


In [13]:
from ligandcontacts import find_consensus_ligand_contacts
from pocket import select_pocket_by_anchor_overlap

# --- アンカー2: 相同蛋白の共結晶化リガンド(ATP拮抗阻害剤)の接触残基 ---
# モチーフだけでは、配列上目立つ保存モチーフを持たないヒンジ領域(アデニン環と水素結合する部分)を
# 拾えない。ダウンロード済みの相同蛋白の結晶構造には共結晶化リガンドを含むものが多いため、その
# 接触残基をCDK20の番号にマッピングしてアンカーに加える(閾値・除外リガンドの根拠は
# ligandcontacts.find_consensus_ligand_contactsのdocstring参照)。
cif_paths = sorted(STRUCT_DIR.glob("*/*.cif"))
ligand_contacts = find_consensus_ligand_contacts(cif_paths, target_chain.sequence)
print(
    f"ligand-derived anchor residues (contacted in >= {ligand_contacts.min_count}/{ligand_contacts.n_ligands} "
    f"ligand-bound structures): {ligand_contacts.anchor_resnums}"
)

anchor_resnums = motif_anchor_resnums | set(ligand_contacts.anchor_resnums)
print(
    f"combined ATP-binding site anchor residues (motif + ligand contacts, n={len(anchor_resnums)}): "
    f"{sorted(anchor_resnums)}"
)

# fpocketのスコアではなく、この2つを統合したアンカー残基を最もよく含むポケットをATP結合部位とする。
selection = select_pocket_by_anchor_overlap(pockets, anchor_resnums, target_chain_id)
atp_pocket = selection.pocket
print(
    f"Using pocket {atp_pocket.pocket_id} (fpocket score={atp_pocket.score:.3f}, "
    f"anchor overlap={selection.overlap} {selection.overlap_resnums}) as the ATP-binding site"
)

# fpocketが検出したポケットの全残基ではなく、アンカー(保存モチーフ+相同蛋白のリガンド接触)で
# 裏付けが取れた残基だけをATP結合部位として採用する。fpocketの3D検出だけに基づく残基
# (根拠のない偶然の近接、あるいはAlphaFold予測の低信頼度領域)を紛れ込ませないため。
atp_pocket_resnums = selection.overlap_resnums
print(f"ATP-binding pocket residues (chain {target_chain_id}, n={len(atp_pocket_resnums)}): {atp_pocket_resnums}")


02:52:21 [ligandcontacts.consensus] scanning 30 structure(s) for co-crystallized ligands ...
02:52:21 [seqalign.pairwise] align_to_reference: 160 substitution(s), 8 gap(s) found (identity=44.6%, coverage=83.5%)
02:52:21 [ligandcontacts.consensus] [1/30] 1GZ8: ligand MBP (17 atoms), 15 contact residue(s) -> reference resnum [10, 11, 12, 13, 18, 31, 33, 65, 81, 82, 83, 84, 131, 134, 144]
02:52:21 [seqalign.pairwise] align_to_reference: 159 substitution(s), 8 gap(s) found (identity=44.8%, coverage=83.2%)
02:52:21 [ligandcontacts.consensus] [2/30] 2R3I: ligand SCF (24 atoms), 20 contact residue(s) -> reference resnum [10, 11, 12, 13, 18, 31, 33, 65, 81, 82, 83, 84, 85, 86, 87, 131, 132, 134, 144, 145]
02:52:21 [seqalign.pairwise] align_to_reference: 157 substitution(s), 8 gap(s) found (identity=44.9%, coverage=82.4%)
02:52:21 [ligandcontacts.consensus] [3/30] 2R3Q: ligand 5SC (26 atoms), 22 contact residue(s) -> reference resnum [8, 10, 11, 12, 13, 18, 20, 31, 33, 65, 81, 82, 83, 84, 85, 8

ligand-derived anchor residues (contacted in >= 5/25 ligand-bound structures): [10, 11, 12, 13, 14, 18, 31, 33, 65, 81, 82, 83, 84, 85, 86, 87, 90, 129, 131, 132, 134, 144, 145]
combined ATP-binding site anchor residues (motif + ligand contacts, n=31): [10, 11, 12, 13, 14, 15, 16, 18, 31, 33, 65, 81, 82, 83, 84, 85, 86, 87, 90, 125, 126, 127, 129, 131, 132, 134, 144, 145, 146, 147, 148]
Using pocket 21 (fpocket score=-0.166, anchor overlap=22 [10, 11, 12, 13, 14, 15, 18, 31, 33, 65, 81, 82, 84, 127, 129, 131, 132, 134, 144, 145, 147, 148]) as the ATP-binding site
ATP-binding pocket residues (chain A, n=22): [10, 11, 12, 13, 14, 15, 18, 31, 33, 65, 81, 82, 84, 127, 129, 131, 132, 134, 144, 145, 147, 148]


In [14]:
from structfit import apply_fit, find_best_chain_for_residues, fit_by_residue_pairs
from structio import write_structure

MIN_POCKET_RESIDUES = 3  # 剛体重ね合わせに最低限必要な対応残基数

cif_paths = sorted(STRUCT_DIR.glob("*/*.cif"))
print(f"aligning {len(cif_paths)} structure(s) onto the AlphaFold model (chain {target_chain_id}) ...")

n_aligned = 0
n_skipped = 0
for i, cif_path in enumerate(cif_paths, start=1):
    entry_name = cif_path.parent.name
    pdb_id = cif_path.stem
    output_path = cif_path.parent / "aligned" / f"{pdb_id}_aligned.cif"
    if output_path.exists():
        print(f"[{i}/{len(cif_paths)}] {entry_name}/{pdb_id}: already exists, skipping")
        n_skipped += 1
        continue

    atoms = parse_structure(cif_path)
    mobile_novo = atoms.select("not water")  # チェーンIDには触れず水分子だけ除く

    best_chain = find_best_chain_for_residues(
        get_chain_sequences(mobile_novo), target_chain.sequence, atp_pocket_resnums
    )
    if best_chain is None or len(best_chain.resnum_pairs) < MIN_POCKET_RESIDUES:
        n_covered = 0 if best_chain is None else len(best_chain.resnum_pairs)
        print(f"[{i}/{len(cif_paths)}] {entry_name}/{pdb_id}: ATP pocket not covered (best {n_covered} residue(s)), skipping")
        n_skipped += 1
        continue

    fit_result = fit_by_residue_pairs(
        cif_path, afdb_output, best_chain.chain_id, target_chain_id, best_chain.resnum_pairs
    )
    apply_fit(fit_result, mobile_novo)
    write_structure(mobile_novo, output_path)

    print(
        f"[{i}/{len(cif_paths)}] {entry_name}/{pdb_id} (chain {best_chain.chain_id}): "
        f"RMSD={fit_result.rmsd:.3f} A ({fit_result.n_residues} pocket residues) -> {output_path}"
    )
    n_aligned += 1

print(f"Done: {n_aligned} aligned, {n_skipped} skipped, {len(cif_paths)} total")


aligning 30 structure(s) onto the AlphaFold model (chain A) ...
[1/30] CDK2_HUMAN/1GZ8: already exists, skipping
[2/30] CDK2_HUMAN/2R3I: already exists, skipping
[3/30] CDK2_HUMAN/2R3Q: already exists, skipping
[4/30] CDK2_HUMAN/2R3R: already exists, skipping
[5/30] CDK2_HUMAN/4EK3: already exists, skipping
[6/30] CDK2_HUMAN/4EK4: already exists, skipping
[7/30] CDK2_HUMAN/4FKL: already exists, skipping
[8/30] CDK2_HUMAN/4FKU: already exists, skipping
[9/30] CDK2_HUMAN/4GCJ: already exists, skipping
[10/30] CDK2_HUMAN/5MHQ: already exists, skipping
[11/30] CDK2_HUMAN/6GUK: already exists, skipping
[12/30] CDK2_HUMAN/6Q3B: already exists, skipping
[13/30] CDK2_HUMAN/6Q3C: already exists, skipping
[14/30] CDK2_HUMAN/6Q3F: already exists, skipping
[15/30] CDK2_HUMAN/6Q48: already exists, skipping
[16/30] CDK2_HUMAN/6Q49: already exists, skipping
[17/30] CDK2_HUMAN/6Q4A: already exists, skipping
[18/30] CDK2_HUMAN/6Q4B: already exists, skipping
[19/30] CDK2_HUMAN/6Q4D: already exists, skip

## 6. 化合物ごとの活性値(pChEMBL)を集計する

セクション2でランキングした蛋白(ChEMBL targetが見つかったもの)について、ChEMBLの活性データを
化合物(構造標準化済み)×標的蛋白の単位で集計し、pChEMBL値のmedian/mean/standard deviation/個数を求める。
median(代表値) >= 9.0(高活性)の組み合わせだけを抽出する。


In [15]:
from chembl import collect_standardized_activities

ACTIVITY_RECORDS_CACHE = OUTDIR / "activity_records.pkl"

if ACTIVITY_RECORDS_CACHE.exists():
    print(f"Using cached activity records: {ACTIVITY_RECORDS_CACHE}")
    activity_records_df = pd.DataFrame(pd.read_pickle(ACTIVITY_RECORDS_CACHE))
else:
    targets_with_chembl = ranking_df.dropna(subset=["chembl_target_id"])
    activity_records_df = collect_standardized_activities(targets_with_chembl, CHEMBL_DB)
    activity_records_df.to_pickle(ACTIVITY_RECORDS_CACHE)
    print(f"Saved {len(activity_records_df)} activity records to {ACTIVITY_RECORDS_CACHE}")


Using cached activity records: data/cdk20_investigation/activity_records.pkl


In [16]:
from chembl import select_high_potency_compounds, summarize_compound_target_activity

PCHEMBL_MEDIAN_CUTOFF = 9.0

print(f"activity records (standardized SMILES, pChEMBL available): {len(activity_records_df)}")

activity_summary_df = summarize_compound_target_activity(activity_records_df)
activity_summary_df.to_csv(OUTDIR / "activity_summary.csv", index=False)
print(f"compound x target pairs: {len(activity_summary_df)}")

high_potency_df = select_high_potency_compounds(
    activity_summary_df, potency_col="median", potency_cutoff=PCHEMBL_MEDIAN_CUTOFF
)
high_potency_df.to_csv(OUTDIR / "high_potency_compounds.csv", index=False)
print(f"compound x target pairs with median pChEMBL >= {PCHEMBL_MEDIAN_CUTOFF}: {len(high_potency_df)}")

high_potency_display_df = high_potency_df.copy()
for col in ["median", "mean", "std"]:
    high_potency_display_df[col] = high_potency_display_df[col].round(2)
with pd.option_context("display.max_colwidth", None):
    display(high_potency_display_df)


02:52:26 [chembl.aggregate] 45/7272 compounds pass the potency/mol_weight filter


activity records (standardized SMILES, pChEMBL available): 9170
compound x target pairs: 7272
compound x target pairs with median pChEMBL >= 9.0: 45


,smiles,accession,entry_name,median,mean,std,count
0,COc1ccc(F)cc1-c1cc(NC(=O)Cc2cccnc2)ncn1,P50750,CDK9_HUMAN,10.82,10.82,NaN,1
1,COc1ccc(F)cc1-c1cc(NC(=O)C2CCCNC2)ncn1,P50750,CDK9_HUMAN,10.74,10.74,NaN,1
2,COc1ccc(F)cc1-c1cc(NC(=O)Cc2ccncc2)ncn1,P50750,CDK9_HUMAN,10.54,10.54,NaN,1
3,COc1cccc(F)c1-c1cc(NC(=O)C2CCCNC2)ncn1,P50750,CDK9_HUMAN,10.41,10.41,NaN,1
4,COc1ccccc1-c1cc(NC(=O)C2CCC(=O)NC2)ncn1,P50750,CDK9_HUMAN,10.24,10.24,0.68,2
5,NS(=O)(=O)c1ccc(Nc2ncc(C(F)(F)F)c(Nc3ccc4[nH]cnc4c3)n2)cc1,P06493,CDK1_HUMAN,10.00,10.00,NaN,1
6,COc1ccccc1-c1cc(NC(=O)c2ccc(C)c(NS(C)(=O)=O)c2)ncn1,P50750,CDK9_HUMAN,9.94,9.94,NaN,1
7,CC(C)c1cnn2c(NCc3ccccc3-n3cccn3)nc(OC3CCCNC3)nc12,P50613,CDK7_HUMAN,9.89,9.89,NaN,1
8,O=[N+]([O-])c1cccc(Nc2nccc(-c3cnn4ncccc34)n2)c1,P24941,CDK2_HUMAN,9.52,9.52,NaN,1
9,Cn1ncc(-c2nc(N[C@H]3CC[C@H](N)CC3)ncc2F)c1CC1CC1,P50613,CDK7_HUMAN,9.51,9.51,NaN,1


### 6.1 化合物単位に集約する

同じ化合物(標準化SMILES)が複数の標的蛋白に対して活性データを持つ場合があるため、化合物ごとに
グループ化し、テストされた標的数(`target_count`)・最も高い活性(median pChEMBL)を示した標的の
Entry name(`best_target_entry_name`)・その活性値(`best_pchembl_median`)に集約する。


In [17]:
from chembl import rollup_compound_summary, select_high_potency_compounds
from molstd import calc_mol_weight

compound_summary_df = rollup_compound_summary(activity_summary_df)
compound_summary_df = compound_summary_df.rename(columns={"best_median": "best_pchembl_median"})
compound_summary_df["mol_weight"] = compound_summary_df["smiles"].apply(calc_mol_weight)
compound_summary_df = compound_summary_df[
    ["smiles", "mol_weight", "best_target_entry_name", "best_pchembl_median", "target_count"]
]
compound_summary_df.to_csv(OUTDIR / "compound_summary.csv", index=False)
print(f"unique compounds: {len(compound_summary_df)}")

MOL_WEIGHT_MIN, MOL_WEIGHT_MAX = 250, 650

high_potency_compound_df = select_high_potency_compounds(
    compound_summary_df, potency_col="best_pchembl_median", potency_cutoff=PCHEMBL_MEDIAN_CUTOFF,
    mol_weight_col="mol_weight", mol_weight_range=(MOL_WEIGHT_MIN, MOL_WEIGHT_MAX),
)
high_potency_compound_df.to_csv(OUTDIR / "high_potency_compound_summary.csv", index=False)
print(
    f"unique compounds with best_pchembl_median >= {PCHEMBL_MEDIAN_CUTOFF} "
    f"and {MOL_WEIGHT_MIN} <= mol_weight <= {MOL_WEIGHT_MAX}: {len(high_potency_compound_df)}"
)

high_potency_compound_display_df = high_potency_compound_df.copy()
high_potency_compound_display_df["best_pchembl_median"] = high_potency_compound_display_df["best_pchembl_median"].round(2)
high_potency_compound_display_df["mol_weight"] = high_potency_compound_display_df["mol_weight"].round(2)

# 表示だけ全行・SMILES非トランケートにする。オプションはこのセル内に限定する
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(high_potency_compound_display_df)


02:52:27 [chembl.aggregate] 40/5003 compounds pass the potency/mol_weight filter


unique compounds: 5003
unique compounds with best_pchembl_median >= 9.0 and 250 <= mol_weight <= 650: 40


,smiles,mol_weight,best_target_entry_name,best_pchembl_median,target_count
0,COc1ccc(F)cc1-c1cc(NC(=O)Cc2cccnc2)ncn1,338.34,CDK9_HUMAN,10.82,1
1,COc1ccc(F)cc1-c1cc(NC(=O)C2CCCNC2)ncn1,330.36,CDK9_HUMAN,10.74,1
2,COc1ccc(F)cc1-c1cc(NC(=O)Cc2ccncc2)ncn1,338.34,CDK9_HUMAN,10.54,1
3,COc1cccc(F)c1-c1cc(NC(=O)C2CCCNC2)ncn1,330.36,CDK9_HUMAN,10.41,1
4,COc1ccccc1-c1cc(NC(=O)C2CCC(=O)NC2)ncn1,326.36,CDK9_HUMAN,10.24,1
5,NS(=O)(=O)c1ccc(Nc2ncc(C(F)(F)F)c(Nc3ccc4[nH]cnc4c3)n2)cc1,449.42,CDK1_HUMAN,10.00,1
6,COc1ccccc1-c1cc(NC(=O)c2ccc(C)c(NS(C)(=O)=O)c2)ncn1,412.47,CDK9_HUMAN,9.94,1
7,CC(C)c1cnn2c(NCc3ccccc3-n3cccn3)nc(OC3CCCNC3)nc12,432.53,CDK7_HUMAN,9.89,4
8,O=[N+]([O-])c1cccc(Nc2nccc(-c3cnn4ncccc34)n2)c1,333.31,CDK2_HUMAN,9.52,1
9,Cn1ncc(-c2nc(N[C@H]3CC[C@H](N)CC3)ncc2F)c1CC1CC1,344.44,CDK7_HUMAN,9.51,2


## 7. ATP結合部位残基をフレキシブルにしたドッキング

セクション5で特定したATP結合部位周辺残基(`atp_pocket_resnums`)を全てフレキシブル(可動側鎖)にした状態で、
セクション6.1で集計した「他のCDKで高活性を示した化合物」(`high_potency_compound_df`、40化合物)を
AlphaFold予測構造(CDK20)にドッキングする。側鎖に回転可能な結合を持たないGly/Alaはフレキシブル指定から
除外する(meekoは可動原子がない残基を指定されても自動的にスキップするが、意図を明確にするため事前に除く)。

19残基(Gly/Ala除く)をフレキシブルにすると探索空間が非常に大きくなる。実測では、hinge付近の3残基のみを
フレキシブルにした場合はリガンド1件・Vina exhaustiveness=8で約10秒だったのに対し、19残基全てをフレキシブル
にすると1件が3分経っても終わらなかった。生物学的な網羅性を優先してこの19残基全てを採用するため、
40化合物では数時間規模の実行時間を見込む。そのため各セルは出力ファイルが既に存在する場合はスキップする
構成にしてあり、バックグラウンドでの分割実行・中断再開を前提とする。

エンジンはAutoDock Vina(`docking`、vinaは`rdkit`とのBoost.Pythonビルド競合を避けるため専用conda環境
`vina`で起動)。リガンド/受容体のPDBQT変換はmeeko(`ligandprep.prepare_ligand_pdbqt`/
`docking.prepare_flexible_receptor`)。


In [18]:
from proteinprep import repair_structure
from docking import prepare_flexible_receptor

DOCK_DIR = OUTDIR / "docking"
DOCK_DIR.mkdir(parents=True, exist_ok=True)

# 水素付加なし(dock用途)で欠損原子を補完する。AlphaFold予測構造は基本的に欠損がないはずだが、
# receptor prepの標準フローに合わせておく。
repaired_receptor_path = DOCK_DIR / "cdk20_repaired.pdb"
if repaired_receptor_path.exists():
    print(f"{repaired_receptor_path}: already exists, skipping")
else:
    repair_structure(afdb_output, repaired_receptor_path, ph=None)
    print(f"Repaired receptor -> {repaired_receptor_path}")

# Gly/Alaは側鎖に回転可能な結合を持たないためフレキシブル指定から除外する
_NO_ROTATABLE_SIDECHAIN = {"G", "A"}
flexible_resnums = [r for r in atp_pocket_resnums if target_chain.sequence[r - 1] not in _NO_ROTATABLE_SIDECHAIN]
print(f"flexible residues ({len(flexible_resnums)}/{len(atp_pocket_resnums)}, Gly/Ala除く): {flexible_resnums}")

flex_receptor = prepare_flexible_receptor(
    repaired_receptor_path,
    [(target_chain_id, r) for r in flexible_resnums],
    DOCK_DIR / "cdk20",
)
print(flex_receptor)


data/cdk20_investigation/docking/cdk20_repaired.pdb: already exists, skipping
flexible residues (15/22, Gly/Ala除く): [10, 12, 15, 18, 33, 65, 81, 82, 84, 127, 129, 132, 134, 145, 148]


02:52:30 [docking.receptor] Prepared flexible receptor: data/cdk20_investigation/docking/cdk20_repaired.pdb (15 flexible residue(s)) -> data/cdk20_investigation/docking/cdk20_rigid.pdbqt, data/cdk20_investigation/docking/cdk20_flex.pdbqt, data/cdk20_investigation/docking/cdk20.json


FlexReceptor(rigid_pdbqt=PosixPath('data/cdk20_investigation/docking/cdk20_rigid.pdbqt'), flex_pdbqt=PosixPath('data/cdk20_investigation/docking/cdk20_flex.pdbqt'), polymer_json=PosixPath('data/cdk20_investigation/docking/cdk20.json'), n_flexible_residues=15)


In [19]:
from docking import calc_search_box

# 探索ボックスはATP結合部位周辺残基(atp_pocket_resnums、フレキシブル化の有無に関わらず全て)の
# CA原子を包含するよう計算する。paddingは既定の4Aのまま(大きくしすぎるとフレキシブル残基の探索と
# 相まって余計に遅くなる)。
pocket_resnum_selection = " ".join(str(r) for r in atp_pocket_resnums)
pocket_ca = target_atoms.select(f"name CA and chain {target_chain_id} and resnum {pocket_resnum_selection}")
box_center, box_size = calc_search_box(pocket_ca.getCoords())
print(f"search box: center={tuple(round(c, 2) for c in box_center)}, size={tuple(round(s, 2) for s in box_size)} A")


search box: center=(-4.73, 5.84, 10.36), size=(24.78, 24.04, 26.32) A


In [20]:
from ligandprep import prepare_ligand_pdbqt

LIGAND_DIR = DOCK_DIR / "ligands"
high_potency_compound_df = high_potency_compound_df.assign(
    ligand_name=[f"compound_{i:03d}" for i in range(1, len(high_potency_compound_df) + 1)]
)

total = len(high_potency_compound_df)
for i, row in enumerate(high_potency_compound_df.itertuples(), start=1):
    out_path = LIGAND_DIR / f"{row.ligand_name}.pdbqt"
    if out_path.exists():
        print(f"[{i}/{total}] {row.ligand_name}: already exists, skipping")
        continue
    prepare_ligand_pdbqt(row.smiles, row.ligand_name, out_path)
    print(f"[{i}/{total}] {row.ligand_name}: {row.smiles} -> {out_path}")


[1/40] compound_001: already exists, skipping
[2/40] compound_002: already exists, skipping
[3/40] compound_003: already exists, skipping
[4/40] compound_004: already exists, skipping
[5/40] compound_005: already exists, skipping
[6/40] compound_006: already exists, skipping
[7/40] compound_007: already exists, skipping
[8/40] compound_008: already exists, skipping
[9/40] compound_009: already exists, skipping
[10/40] compound_010: already exists, skipping
[11/40] compound_011: already exists, skipping
[12/40] compound_012: already exists, skipping
[13/40] compound_013: already exists, skipping
[14/40] compound_014: already exists, skipping
[15/40] compound_015: already exists, skipping
[16/40] compound_016: already exists, skipping
[17/40] compound_017: already exists, skipping
[18/40] compound_018: already exists, skipping
[19/40] compound_019: already exists, skipping
[20/40] compound_020: already exists, skipping
[21/40] compound_021: already exists, skipping
[22/40] compound_022: 

In [21]:
from docking import parse_vina_output, run_vina

POSES_DIR = DOCK_DIR / "poses"
DOCKING_EXHAUSTIVENESS = 8
DOCKING_NUM_MODES = 9

dock_rows = []
total = len(high_potency_compound_df)
for i, row in enumerate(high_potency_compound_df.itertuples(), start=1):
    ligand_pdbqt = LIGAND_DIR / f"{row.ligand_name}.pdbqt"
    out_path = POSES_DIR / f"{row.ligand_name}_docked.pdbqt"
    if out_path.exists():
        print(f"[{i}/{total}] {row.ligand_name}: already docked, skipping")
    else:
        print(f"[{i}/{total}] docking {row.ligand_name} ({row.smiles}) ...")
        run_vina(
            rigid_pdbqt=flex_receptor.rigid_pdbqt,
            ligand_pdbqt=ligand_pdbqt,
            center=box_center,
            size=box_size,
            output_path=out_path,
            flex_pdbqt=flex_receptor.flex_pdbqt,
            exhaustiveness=DOCKING_EXHAUSTIVENESS,
            num_modes=DOCKING_NUM_MODES,
        )

    # 既にドッキング済みの場合もファイルからスコアを読み直す(このセルを再実行した場合も
    # 同じロジックで結果テーブルを組み立てられるようにするため)
    poses = parse_vina_output(out_path)
    dock_rows.append({
        "ligand_name": row.ligand_name,
        "smiles": row.smiles,
        "best_target_entry_name": row.best_target_entry_name,
        "best_pchembl_median": row.best_pchembl_median,
        "vina_best_affinity": poses[0].affinity,
        "n_poses": len(poses),
        "output_path": str(out_path),
    })
    print(f"    -> best affinity {poses[0].affinity:.2f} kcal/mol")

dock_results_df = pd.DataFrame(dock_rows).sort_values("vina_best_affinity").reset_index(drop=True)
dock_results_df.to_csv(OUTDIR / "docking_results.csv", index=False)
dock_results_df


02:52:39 [docking.vina] Running Vina (env=vina): data/cdk20_investigation/docking/ligands/compound_018.pdbqt -> data/cdk20_investigation/docking/poses/compound_018_docked.pdbqt


[1/40] compound_001: already docked, skipping
    -> best affinity -7.71 kcal/mol
[2/40] compound_002: already docked, skipping
    -> best affinity -8.10 kcal/mol
[3/40] compound_003: already docked, skipping
    -> best affinity -7.51 kcal/mol
[4/40] compound_004: already docked, skipping
    -> best affinity -7.53 kcal/mol
[5/40] compound_005: already docked, skipping
    -> best affinity -7.78 kcal/mol
[6/40] compound_006: already docked, skipping
    -> best affinity -9.61 kcal/mol
[7/40] compound_007: already docked, skipping
    -> best affinity -8.30 kcal/mol
[8/40] compound_008: already docked, skipping
    -> best affinity -8.64 kcal/mol
[9/40] compound_009: already docked, skipping
    -> best affinity -7.74 kcal/mol
[10/40] compound_010: already docked, skipping
    -> best affinity -7.12 kcal/mol
[11/40] compound_011: already docked, skipping
    -> best affinity -8.10 kcal/mol
[12/40] compound_012: already docked, skipping
    -> best affinity -8.12 kcal/mol
[13/40] compo

03:00:28 [docking.vina] Done: data/cdk20_investigation/docking/ligands/compound_018.pdbqt -> best affinity -9.22 kcal/mol (9 pose(s))
03:00:28 [docking.vina] Running Vina (env=vina): data/cdk20_investigation/docking/ligands/compound_019.pdbqt -> data/cdk20_investigation/docking/poses/compound_019_docked.pdbqt


    -> best affinity -9.22 kcal/mol
[19/40] docking compound_019 (CN1CCC(c2c(O)cc(O)c3c(=O)cc(-c4ccccc4Cl)oc23)C(O)C1) ...


03:06:50 [docking.vina] Done: data/cdk20_investigation/docking/ligands/compound_019.pdbqt -> best affinity -9.49 kcal/mol (9 pose(s))
03:06:50 [docking.vina] Running Vina (env=vina): data/cdk20_investigation/docking/ligands/compound_020.pdbqt -> data/cdk20_investigation/docking/poses/compound_020_docked.pdbqt


...

05:35:01 [docking.vina] Done: data/cdk20_investigation/docking/ligands/compound_039.pdbqt -> best affinity -8.83 kcal/mol (9 pose(s))
05:35:01 [docking.vina] Running Vina (env=vina): data/cdk20_investigation/docking/ligands/compound_040.pdbqt -> data/cdk20_investigation/docking/poses/compound_040_docked.pdbqt


    -> best affinity -8.83 kcal/mol
[40/40] docking compound_040 (COCCOCCOCCOCCN(C)S(=O)(=O)c1ccc(N/C=C2\C(=O)Nc3ccc4ncsc4c32)cc1) ...


05:46:34 [docking.vina] Done: data/cdk20_investigation/docking/ligands/compound_040.pdbqt -> best affinity -6.59 kcal/mol (9 pose(s))


    -> best affinity -6.59 kcal/mol


,ligand_name,smiles,best_target_entry_name,best_pchembl_median,vina_best_affinity,n_poses,output_path
0,compound_006,NS(=O)(=O)c1ccc(Nc2ncc(C(F)(F)F)c(Nc3ccc4[nH]c...,CDK1_HUMAN,10.000,-9.609,9,data/cdk20_investigation/docking/poses/compoun...
1,compound_019,CN1CCC(c2c(O)cc(O)c3c(=O)cc(-c4ccccc4Cl)oc23)C...,CDK9_HUMAN,9.200,-9.493,9,data/cdk20_investigation/docking/poses/compoun...
2,compound_036,CN1CCN(c2cccc(Nc3ncc4c(n3)-c3c(c(C(N)=O)nn3C)C...,CDK2_HUMAN,9.000,-9.279,9,data/cdk20_investigation/docking/poses/compoun...
3,compound_018,NC(COc1cncc(-c2ccc3c(c2)C(=Cc2ccc[nH]2)C(=O)N3...,CDK5_HUMAN,9.200,-9.224,9,data/cdk20_investigation/docking/poses/compoun...
4,compound_020,CCNCc1cncc(-c2cnc3[nH]nc(-c4nc5cc(F)ccc5[nH]4)...,CDK1_HUMAN,9.150,-9.095,9,data/cdk20_investigation/docking/poses/compoun...
5,compound_025,C=CC(=O)N1CCc2nc(Nc3ncc(F)c(-c4cc(F)c5nc(C)n(C...,CDK9_HUMAN,9.040,-9.080,9,data/cdk20_investigation/docking/poses/compoun...
6,compound_021,CCNCc1cncc(-c2cnc3[nH]nc(-c4nc5cc(OC)ccc5[nH]4...,CDK1_HUMAN,9.100,-8.905,9,data/cdk20_investigation/docking/poses/compoun...
7,compound_039,Fc1cc(F)cc(Nc2nccc(-c3cnn4ncccc34)n2)c1,CDK2_HUMAN,9.000,-8.832,9,data/cdk20_investigation/docking/poses/compoun...
8,compound_026,FC(F)(F)c1cnc(Nc2ccc3[nH]cnc3c2)nc1Nc1ccccc1Cl,CDK1_HUMAN,9.025,-8.693,9,data/cdk20_investigation/docking/poses/compoun...
9,compound_008,CC(C)c1cnn2c(NCc3ccccc3-n3cccn3)nc(OC3CCCNC3)nc12,CDK7_HUMAN,9.890,-8.636,9,data/cdk20_investigation/docking/poses/compoun...


### 7.1 蛋白コンフォメーション・リガンドポーズの保存

Vinaの出力(`_docked.pdbqt`)自体にはリガンドポーズと可動側鎖の座標は含まれるが、受容体のリジッド部分
(`--receptor`に渡した`_rigid.pdbqt`、ドッキング中は不変)は含まれない。後でインタラクションを解析したり
MDの初期状態として使ったりできるように、`docking.export_docked_poses`で各化合物の最良ポーズ(モード1)
について、受容体フルコンフォメーション(リジッド部分+ドッキング後の可動側鎖、標準PDB形式・水素付き)と
リガンドポーズ(結合次数を復元したSDF)を組み合わせて書き出す。全モードが必要な場合は`modes=None`にする。


In [22]:
from docking import export_docked_poses

EXPORT_DIR = DOCK_DIR / "exported_poses"

total = len(high_potency_compound_df)
for i, row in enumerate(high_potency_compound_df.itertuples(), start=1):
    docked_pdbqt = POSES_DIR / f"{row.ligand_name}_docked.pdbqt"
    exported_marker = EXPORT_DIR / f"{row.ligand_name}_mode1_receptor.pdb"
    if exported_marker.exists():
        print(f"[{i}/{total}] {row.ligand_name}: already exported, skipping")
        continue
    exported = export_docked_poses(
        polymer_json=flex_receptor.polymer_json,
        vina_output_pdbqt=docked_pdbqt,
        output_dir=EXPORT_DIR,
        name=row.ligand_name,
        modes=[1],  # 最良ポーズのみ。全モードが必要な場合はmodes=Noneにする
    )
    pose = exported[0]
    print(f"[{i}/{total}] {row.ligand_name}: exported -> {pose.receptor_pdb}, {pose.ligand_sdf}")


05:46:35 [docking.export] Exported 1 pose(s) for compound_001 -> data/cdk20_investigation/docking/exported_poses


[1/40] compound_001: exported -> data/cdk20_investigation/docking/exported_poses/compound_001_mode1_receptor.pdb, data/cdk20_investigation/docking/exported_poses/compound_001_mode1_ligand.sdf


05:46:36 [docking.export] Exported 1 pose(s) for compound_002 -> data/cdk20_investigation/docking/exported_poses


[2/40] compound_002: exported -> data/cdk20_investigation/docking/exported_poses/compound_002_mode1_receptor.pdb, data/cdk20_investigation/docking/exported_poses/compound_002_mode1_ligand.sdf


05:46:37 [docking.export] Exported 1 pose(s) for compound_003 -> data/cdk20_investigation/docking/exported_poses


...

[38/40] compound_038: exported -> data/cdk20_investigation/docking/exported_poses/compound_038_mode1_receptor.pdb, data/cdk20_investigation/docking/exported_poses/compound_038_mode1_ligand.sdf


05:47:18 [docking.export] Exported 1 pose(s) for compound_039 -> data/cdk20_investigation/docking/exported_poses


[39/40] compound_039: exported -> data/cdk20_investigation/docking/exported_poses/compound_039_mode1_receptor.pdb, data/cdk20_investigation/docking/exported_poses/compound_039_mode1_ligand.sdf


05:47:19 [docking.export] Exported 1 pose(s) for compound_040 -> data/cdk20_investigation/docking/exported_poses


[40/40] compound_040: exported -> data/cdk20_investigation/docking/exported_poses/compound_040_mode1_receptor.pdb, data/cdk20_investigation/docking/exported_poses/compound_040_mode1_ligand.sdf


## 8. Boltz-2によるCDK20構造予測(DFG-in / DFG-out)

セクション0でダウンロードしたAlphaFold予測構造とは別に、[Boltz-2](https://github.com/jwohlwend/boltz)
(AlphaFold3系のオープンソース構造予測モデル)でCDK20の構造を予測する。ローカルGPU(VRAM 6GB)では
不足するため、NVIDIAのホスト型API(`folding`、認証は`.env`の`NVIDIA_API_KEY`)を使う。

1. 通常のDFG-in構造(テンプレートなし)
2. DFGをテンプレートで誘導したDFG-out構造。CDK20自体にDFG-out構造は無いため(CDK系はType I阻害剤が
   主流でDFG-out例が乏しい)、**CDK2**(セクション2のBLAST検索でも上位に来た近縁蛋白、配列同一性
   ~44-45%)のType II阻害剤共結晶構造(PDB [5A14](https://www.rcsb.org/structure/5A14)、
   Cavazzoli et al., ACS Chem. Biol. 2015「初めて報告されたCDK2のType II阻害剤共結晶構造」、
   DFG-out)をテンプレートに使う。同じCMGCグループでもCDK2はさらに同じCDKサブファミリーであり、
   別サブファミリーのp38 MAPK等より配列的に近いテンプレート。

MSAは1回検索し、両方の予測で使い回す(テンプレートの効果だけを比較できるように)。


In [23]:
from dotenv import load_dotenv

load_dotenv()  # PharmoForgeリポジトリ直下の.envからNVIDIA_API_KEYを読み込む

from folding import search_msa

BOLTZ_DIR = OUTDIR / "boltz"
BOLTZ_DIR.mkdir(parents=True, exist_ok=True)

msa_path = BOLTZ_DIR / "cdk20.a3m"
if msa_path.exists():
    print(f"{msa_path}: already exists, skipping")
else:
    search_msa(sequence, msa_path)


data/cdk20_investigation/boltz/cdk20.a3m: already exists, skipping


In [24]:
from folding import predict_structure

dfg_in_dir = BOLTZ_DIR / "dfg_in"
existing = sorted(dfg_in_dir.glob("cdk20_dfg_in_model*.cif")) if dfg_in_dir.exists() else []
if existing:
    print(f"{dfg_in_dir}: already exists ({len(existing)} model(s)), skipping")
    from folding.boltz import BoltzPrediction
    dfg_in_result = BoltzPrediction(structure_paths=existing, confidence_scores=[], ptm_scores=[])
else:
    dfg_in_result = predict_structure(
        sequence,
        dfg_in_dir,
        "cdk20_dfg_in",
        msa_path=msa_path,
        diffusion_samples=5,
    )
print(dfg_in_result)


data/cdk20_investigation/boltz/dfg_in: already exists (5 model(s)), skipping
BoltzPrediction(structure_paths=[PosixPath('data/cdk20_investigation/boltz/dfg_in/cdk20_dfg_in_model1.cif'), PosixPath('data/cdk20_investigation/boltz/dfg_in/cdk20_dfg_in_model2.cif'), PosixPath('data/cdk20_investigation/boltz/dfg_in/cdk20_dfg_in_model3.cif'), PosixPath('data/cdk20_investigation/boltz/dfg_in/cdk20_dfg_in_model4.cif'), PosixPath('data/cdk20_investigation/boltz/dfg_in/cdk20_dfg_in_model5.cif')], confidence_scores=[], ptm_scores=[])


### 8.1 DFG-outテンプレート構造(CDK2, PDB 5A14)を取得する


In [ ]:
from rcsb import fetch_structure
from folding import StructureTemplate

dfg_out_template_path = BOLTZ_DIR / "5A14_template.cif"
if dfg_out_template_path.exists():
    print(f"{dfg_out_template_path}: already exists, skipping")
else:
    fetch_structure("5A14", dfg_out_template_path)

# PDB 5A14のchain AがCDK2(蛋白鎖)。共結晶化リガンドLQ5(Type II阻害剤)がDFG-outを安定化している。
dfg_out_template = StructureTemplate(structure_path=dfg_out_template_path, chain_id="A", name="cdk2_5a14_dfgout")


In [ ]:
dfg_out_dir = BOLTZ_DIR / "dfg_out"
existing = sorted(dfg_out_dir.glob("cdk20_dfg_out_model*.cif")) if dfg_out_dir.exists() else []
if existing:
    print(f"{dfg_out_dir}: already exists ({len(existing)} model(s)), skipping")
    from folding.boltz import BoltzPrediction
    dfg_out_result = BoltzPrediction(structure_paths=existing, confidence_scores=[], ptm_scores=[])
else:
    dfg_out_result = predict_structure(
        sequence,
        dfg_out_dir,
        "cdk20_dfg_out",
        msa_path=msa_path,
        templates=[dfg_out_template],
        diffusion_samples=5,
    )
print(dfg_out_result)


### 8.2 DFGモチーフの位置を比較する

DFGモチーフ(`kinasemotifs.find_kinase_motifs`で検出したCDK20のPhe146)の位置を、フレーム非依存な
原子間距離(触媒Lys33・hinge領域resnum 65からの距離)でDFG-in/DFG-out予測間で比較する。DFG-outへの
誘導が効いていれば、Phe146の側鎖(CZ原子)の位置がDFG-in予測から明確にシフトするはず。


In [ ]:
import numpy as np
from structio import parse_structure

dfg_phe_resnum = motifs.dfg[0] + 1  # D-F-Gの"F"
lys_resnum = motifs.catalytic_lys
hinge_resnum = 65

def measure_dfg_geometry(cif_path, chain_id="A", phe_resnum=dfg_phe_resnum, lys_resnum_=lys_resnum):
    atoms = parse_structure(cif_path)
    phe_cz = atoms.select(f"chain {chain_id} and resnum {phe_resnum} and name CZ")
    lys_nz = atoms.select(f"chain {chain_id} and resnum {lys_resnum_} and name NZ")
    return float(np.linalg.norm(phe_cz.getCoords()[0] - lys_nz.getCoords()[0]))

print(f"{'model':<8} {'DFG-in: Phe-Lys':>16}   {'DFG-out(CDK2 template): Phe-Lys':>32}")
for i, (in_path, out_path) in enumerate(zip(dfg_in_result.structure_paths, dfg_out_result.structure_paths), start=1):
    d_in = measure_dfg_geometry(in_path)
    d_out = measure_dfg_geometry(out_path)
    print(f"model{i:<3} {d_in:>16.2f}   {d_out:>32.2f}")

# 参照値: CDK2(5A14)自身の実際のDFG-out構造でのPhe(CZ)-Lys(NZ)距離("正解"の目安)。
# 5A14はN末端側に欠損があるためauth番号にギャップがあるが、DFGのPhe・触媒Lysの実際のauth番号は
# たまたまCDK20と同じ(Phe146・Lys33)。
d_ref = measure_dfg_geometry(dfg_out_template_path)
print(f"\n参照(CDK2 5A14自身、真のDFG-out): Phe-Lys = {d_ref:.2f} A")


**結果(実測)**: DFG-in予測とDFG-out(CDK2 5A14テンプレート誘導)予測の間で、Phe146の位置
(触媒Lys33からの距離)はサンプル間ばらつきの範囲内でほぼ同じだった(いずれも約10.3〜12.4A)。
一方、CDK2(5A14)自身の実際のDFG-out構造での同じ指標は8.91Aであり、CDK20の予測群
(DFG-in・DFG-out試行とも)とは明確に異なる値を示している。つまり**CDK20のDFG-out予測は、
より近縁(同じCDKサブファミリー)なテンプレートに差し替えても、実際のDFG-out状態には到達しなかった**。

さらに以前p38 MAPK(PDB 1KV1、別のCMGCサブファミリー)をテンプレートにした場合も同様に失敗して
いる(詳細は[FOLDING_PROMPT.md](../folding/FOLDING_PROMPT.md#テンプレート誘導の検証)参照)。
**配列的に近いテンプレートに変えても結果が変わらなかった**ことから、テンプレート選択の問題では
なく、NVIDIAホスト型APIのテンプレート機構自体(OSS版`boltz`の`force`/`threshold`のような拘束
ポテンシャルを持たない、モデルのテンプレート注意機構によるソフトな誘導のみ)が、DFGモチーフの
フリップのような大きな構造変化を起こすには不十分である可能性が高いと考えられる。

より確実にDFG-outを誘導したい場合、次の選択肢が考えられる(いずれも未実施):
- OSS版`boltz`をローカルGPUで実行し`force`/`threshold`拘束を使う(VRAM不足のため要検討)
- `recycling_steps`を増やす、複数テンプレートを組み合わせる等のパラメータ探索
- テンプレートベースではなく、拘束(`contact`等、蛋白内残基間距離を強制できるものがあれば)で
  直接DFG-Pheの位置を誘導する(現状NVIDIAホスト型APIには蛋白内残基間の距離拘束は無い)
